In Python, functions are ordinary objects: you can store them in variables, pass them as arguments, and return them from other functions. This enables a clean *functional* style that pairs well with numerical work.

::: {.callout-tip}
## Why learn this?
In Python, functions are just values you can pass around — which is what lets you write *generic* tools: one integrator that works for any spectrum, one solver that accepts any right-hand side. Type hints keep that flexibility from turning into chaos.

- **Imagine you need** an integrator that works for *any* function you throw at it — just pass the function in.
- **Imagine you need** to always call `reynolds(...)` with the same viscosity — `functools.partial` bakes it in once.
:::

## Functions are first-class objects
A function can be assigned, put in a container, and passed around like any value.

In [1]:
import numpy as np

def csv(x):      return ', '.join(map(str, x))
def total(x):    return sum(x)

# store functions in a dict and pick one at runtime
ops = {'join': csv, 'sum': total}
data = [1, 2, 3, 4]
print(ops['join'](data))
print(ops['sum'](data))

# a function that TAKES a function (a 'higher-order' function)
def apply_twice(f, x):
    return f(f(x))
print(apply_twice(lambda v: v * 3, 2))

1, 2, 3, 4
10
18


## Lambdas, `map`, and `filter`
A `lambda` is a small anonymous function. `map`/`filter` apply a function across an iterable lazily. For elementwise math on arrays, prefer NumPy vectorisation; `map`/`lambda` shine for light glue and custom keys.

In [2]:
nums = range(1, 8)

squared = list(map(lambda n: n ** 2, nums))
odd = list(filter(lambda n: n % 2, nums))
print('map   :', squared)
print('filter:', odd)

# lambdas as sort keys
cases = [('a', 5000), ('b', 200), ('c', 900)]
print('sorted:', sorted(cases, key=lambda t: t[1]))

# the equivalent comprehensions (often more readable)
print('comp  :', [n ** 2 for n in nums])

map   : [1, 4, 9, 16, 25, 36, 49]
filter: [1, 3, 5, 7]
sorted: [('b', 200), ('c', 900), ('a', 5000)]
comp  : [1, 4, 9, 16, 25, 36, 49]


**Imagine you need** a specialised version of a general function — always the same viscosity, always base 2. This is super easy in Python using `functools.partial`.

## `functools`: reduce and partial
`reduce` folds an iterable to a single value; `partial` pre-fills some arguments to make a new, specialised function.

In [3]:
from functools import reduce, partial

product = reduce(lambda a, b: a * b, [1, 2, 3, 4], 1)
print('reduce (4!):', product)

def power(base, exponent):
    return base ** exponent

square = partial(power, exponent=2)   # fix exponent=2
cube = partial(power, exponent=3)
print('square(5):', square(5), '| cube(2):', cube(2))

reduce (4!): 24
square(5): 25 | cube(2): 8


**Imagine you need** to hand a module to a labmate and have their editor catch mistakes before running anything. This is super easy in Python using *type hints*.

## Type hints
Annotations document the *intended* types of arguments and return values. They do not change runtime behaviour, but they make code self-explanatory and let tools (IDEs, `mypy`) catch mistakes before you run.

In [4]:
from typing import Callable, Optional
from collections.abc import Sequence

def integrate(f: Callable[[float], float],
              a: float, b: float, n: int = 100) -> float:
    """Midpoint-rule integral of f over [a, b]."""
    h = (b - a) / n
    return h * sum(f(a + (i + 0.5) * h) for i in range(n))

def normalise(xs: Sequence[float],
              scale: Optional[float] = None) -> list[float]:
    s = scale if scale is not None else max(abs(x) for x in xs)
    return [x / s for x in xs]

import math
print('integral of sin on [0, pi] ~', round(integrate(math.sin, 0, math.pi), 4))
print('normalised:', normalise([1.0, -2.0, 4.0]))

integral of sin on [0, pi] ~ 2.0001
normalised: [0.25, -0.5, 1.0]


## Self-tests

**1.** Use `map` (or a comprehension) to turn `['1','2','3']` into `[1, 2, 3]` (ints).

In [ ]:
#| code-fold: true
#| code-summary: "Show solution"
#| output: false
print(list(map(int, ['1', '2', '3'])))

**2.** With `functools.partial`, make a function `to_kelvin(celsius)` from a general `def shift(x, by): return x + by` (water freezes at 273.15 K).

In [ ]:
#| code-fold: true
#| code-summary: "Show solution"
#| output: false
from functools import partial
def shift(x, by):
    return x + by
to_kelvin = partial(shift, by=273.15)
print(to_kelvin(25))

**3.** Add type hints to `def mean(xs): return sum(xs)/len(xs)` for a sequence of floats returning a float.

::: {.callout-tip collapse="true"}
## Solution
```python
from collections.abc import Sequence
def mean(xs: Sequence[float]) -> float:
    return sum(xs) / len(xs)
```
:::

## Turbulence in practice: TKE from a spectrum

The turbulent kinetic energy is the integral of the spectrum, $k=\int_0^\infty E(\kappa)\,\mathrm d\kappa$. Reuse the functional `integrate` from above, passing a model spectrum *as a function*.

In [7]:
import numpy as np

def model_spectrum(kappa, C=1.5, eps=1.0, k_max=1000.0):
    # inertial range with a high-wavenumber cutoff (finite integral)
    return C * eps ** (2 / 3) * kappa ** (-5 / 3) if kappa <= k_max else 0.0

tke = integrate(model_spectrum, 1.0, 1000.0, n=5000)
print(f'TKE = \u222b E dk = {tke:.3f} m^2/s^2')

TKE = ∫ E dk = 2.223 m^2/s^2


**Self-test.** Use `functools.reduce` to sum the per-band energies `[0.5, 0.3, 0.15, 0.05]` into the total.

In [ ]:
#| code-fold: true
#| code-summary: "Show solution"
#| output: false
from functools import reduce
print(reduce(lambda a, b: a + b, [0.5, 0.3, 0.15, 0.05]))